# 04: Turn your calculation into a cloud job

Start with the circuit quick start below. For a complete Python program, continue with the script workflow.
You need a configured Jiuding development environment with shared storage. See [Setup](../../SETUP.md).
The remote cells do nothing until you explicitly enable submission.

## Choose where receipts and results will live

Use the workshop outputs for the local exercise. When submission is enabled, the notebook automatically stages the program and results on a writable mount inherited by Jiuding jobs.
An image-local folder such as `/opt/flagquantum` is not shared storage. If no configured mount can be verified, submission stops with instructions for the instructor.

In [ ]:
from pathlib import Path

# Support both a full checkout and a deployment containing only the workshop.
search_roots = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
WORKSHOP = next(
    (root / "flagos2026" for root in search_roots if (root / "flagos2026/notebooks").is_dir()),
    None,
)
if WORKSHOP is None:
    raise RuntimeError("Cannot locate flagos2026/notebooks above the current working directory. Open this notebook from the deployed workshop tree.")
repository = WORKSHOP.parent.parent
ROOT = repository if (repository / "pyproject.toml").is_file() else WORKSHOP
OUTPUTS = WORKSHOP / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)
print("Your result folder:", OUTPUTS)


## Circuit quick start: choose resources, submit, retrieve

Use an existing Jiuding private image containing this FlagQuantum submission API. The default CPU task needs no accelerator card or GPU runtime, and no GHCR access is required. Set `JIUDING_IMAGE`, `JIUDING_PROJECT` (`project-set.project`), and `JIUDING_QUEUE` in your environment, or replace the corresponding values below with those supplied by your instructor.
Authentication uses the configured provider credentials, including the base64-encoded AK/SK mount at `/etc/accesskey` on the platform. Do not paste or print credentials in this notebook. A laptop needs configured credentials and access to the Jiuding service; it does not need a local GPU.

Start with `native_target = "jiuding:cpu"` and `devices = 1`: this means one CPU execution device, not one accelerator card. A queue labeled `KUNLUN_R300` can still run a CPU task with zero cards allocated.

Optional GPU execution uses the same API: change `native_target` to `"jiuding:gpu/NVIDIA_A100-SXM4-40GB"`, select a compatible image and queue, and choose `devices = 1` for one card or `devices = 2` for two cards. Allocation does not by itself prove speedup or capacity expansion. This example retrieves a small full state; multi-card execution currently uses forward-only sharding and gathers the full result.

Run with `submit_circuit = False` first, then explicitly enable it when ready. Re-running with an existing receipt is rejected before submission. For a different experiment, choose a new receipt filename.

In [ ]:
import os
import flagquantum as fq

circuit = fq.Circuit(8).h(7).cx(7, 0)
native_target = "jiuding:cpu"
devices = 1
native_image = os.getenv("JIUDING_IMAGE", "")
native_project = os.getenv("JIUDING_PROJECT", "")
native_queue = os.getenv("JIUDING_QUEUE", "")
native_receipt_path = OUTPUTS / "circuit_job.receipt.json"
submit_circuit = False

if submit_circuit:
    if native_receipt_path.exists():
        raise FileExistsError("Receipt already exists. Restore this job instead of submitting again.")
    if not all((native_image, native_project, native_queue)):
        raise ValueError("Configure JIUDING_IMAGE, JIUDING_PROJECT and JIUDING_QUEUE before submitting.")
    job = fq.submit(
        circuit, target=native_target, devices=devices,
        image=native_image, project=native_project, queue=native_queue,
    )
    print("Job ID:", job.id)
    job.save(native_receipt_path)
    print("Receipt:", native_receipt_path)
else:
    print("No circuit job submitted. Choose resources above, then enable submit_circuit.")


Enable `check_circuit` to restore and inspect the saved job, including after a kernel restart. Re-run this cell while queued or running; it never submits a job. A failed job is not retried automatically. If submission succeeds but saving the receipt fails, keep the printed job ID and inspect it in the platform before doing anything else.

In [ ]:
check_circuit = False
if check_circuit:
    job = fq.restore_job(native_receipt_path)
    status = job.status()
    print("Job ID:", job.id, "Status:", status)
    if status == "succeeded":
        import torch

        result = job.result()
        torch.testing.assert_close(result.state, circuit.state())
        print("Probabilities:", result.state.abs().square().tolist())
        print("Runtime:", dict(result.runtime))
        print("Provenance:", dict(result.provenance))
    elif status in {"failed", "cancelled"}:
        print("Inspect the job logs in Jiuding. No retry was submitted.")
    else:
        print("Not complete yet. Re-run this check later; do not submit again.")


## Script workflow: submit a complete Python program

Use the following workflow when your job is a Python script rather than one circuit. It is independent of the circuit quick start above.

## 2. Write the program

This is the Bell calculation you built in notebook 01, now inside a `main()` function.
Jiuding's worker calls `main()` and saves its return value. The function can call other functions and import your shared modules.
For this example, a list of probabilities is easy to save as JSON.

In [ ]:
program_text = """import flagquantum as fq


def main():
    circuit = fq.Circuit(2)
    circuit.h(0)
    circuit.cx(0, 1)
    options = fq.ExecutionOptions(device="cpu", mode="statevector")
    result = fq.run(circuit, options=options)
    state = result.to_statevector().reshape(-1)
    return {
        "device": str(state.device),
        "probabilities": state.abs().square().tolist(),
    }
"""
script_path = OUTPUTS / "my_bell_job.py"
script_path.write_text(program_text)
print("Saved program:", script_path)


## 3. Try your program locally first

`runpy.run_path` loads the file we just wrote. Calling its `main` function lets us catch basic mistakes before using a cloud queue.

In [ ]:
import runpy

program = runpy.run_path(str(script_path))
local_result = program["main"]()
print(local_result)


## 4. Describe the cloud environment

An image contains the software installed in the job container. Ask your instructor for its complete address and Python path.
`JiudingClient` handles authentication and job requests. It does not run a job just because you create it.

In [ ]:
from flagquantum.remote.compute.jiuding import JiudingClient

client = JiudingClient()
target = "jiuding:cpu"
image = ""  # Full image address supplied by your instructor.
image_region = "PRIVATE"  # Use PUBLIC only for a public catalog image.
job_python = "python3"  # Must exist inside that image.
submit_job = False
receipt_pointer = OUTPUTS / "my_bell_job.latest_receipt_path.txt"


## 5. Submit once and save the receipt

A receipt is a JSON file identifying your submission. Keep it so you can check the job after closing this notebook.
Set `submit_job` to `True` in the previous cell when ready. A new private staging directory and receipt path are created automatically. Each deliberate rerun of this submission cell starts a new job; use the progress cell to inspect an existing job instead.
The job uses FlagQuantum installed in the image, not the notebook's source directory. Its shared mounts come from the current development environment.

In [ ]:
if submit_job:
    if not image:
        raise ValueError("Enter the instructor's image address first.")
    import sys
    sys.path.insert(0, str(WORKSHOP / "scripts"))
    from jiuding_paths import stage_job

    shared_script, receipt_path = stage_job(
        program_text, client.workspace().get("storageInfo"),
        preferred=Path("/share/project/flagquantum"),
    )
    receipt_pointer.write_text(str(receipt_path), encoding="utf-8")
    print("Shared program:", shared_script)
    print("Keep this receipt path:", receipt_path)
    receipt = client.submit(
        shared_script,
        target=target,
        image=image,
        image_region=image_region,
        python=job_python,
        receipt=receipt_path,
        cpus=2,
        memory_gib=2,
    )
    print(receipt)
else:
    print("No cloud job submitted. The local program is ready.")


## 6. Check progress and read the result

Enable `check_job` after submitting. Loading the saved receipt works even after a kernel restart.
A timeout means we stopped waiting; it does not mean the job stopped.

In [ ]:
import json

check_job = False
if check_job:
    if not receipt_pointer.is_file():
        raise RuntimeError("No saved submission path. Submit once before checking progress.")
    receipt_path = Path(receipt_pointer.read_text(encoding="utf-8").strip())
    receipt = json.loads(receipt_path.read_text())
    print(client.status(receipt))
    try:
        cloud_result = client.result(receipt, timeout=180)
        print(cloud_result)
        import torch

        torch.testing.assert_close(
            torch.tensor(cloud_result["probabilities"]),
            torch.tensor(local_result["probabilities"]),
        )
    except TimeoutError:
        print("Still waiting. Keep the receipt and check status again later.")


## 7. Stop a job you no longer need

A cancellation request may take time to complete. Check status again until the platform confirms that the job has stopped.

In [ ]:
cancel_job = False
if cancel_job:
    if not receipt_pointer.is_file():
        raise RuntimeError("No saved submission path. There is no recorded job to cancel.")
    receipt_path = Path(receipt_pointer.read_text(encoding="utf-8").strip())
    receipt = json.loads(receipt_path.read_text())
    client.cancel(receipt)
    print(client.status(receipt))


## Try a change

Replace the contents of `program_text` with a small training program based on notebook 03.
Return a dictionary containing its loss history. Save it under a new filename and use a new receipt path for this distinct experiment.

For GPU execution, use a `jiuding:gpu/<model>` target and select CUDA inside the program.
The standalone `scripts/jiuding_job.py` selects the existing GPU Bell example from `--target`; see its `--help` output.